In [2]:
// CELDA 0 — Inicialización
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val spark = SparkSession.builder()
  .appName("Dia23S2-BuenasPracticas")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .getOrCreate()

import spark.implicits._
spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark${spark.version} listo | Scala${scala.util.Properties.versionString}")

✅ Spark4.1.1 listo | Scalaversion 2.13.18


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@11a53024
import spark.implicits._

In [3]:
// CELDA 1 — Crear ficheros CSV
val carpeta = "C:/Curso-Scala/datos/dia23"
Files.createDirectories(Paths.get(carpeta))

val ventasCSV =
  """id,producto,categoria,precio,cantidad,ciudad
    |1,Laptop,Tecnología,1200.0,2,Madrid
    |2,Teclado,Tecnología,45.0,5,Valencia
    |3,Monitor,Tecnología,350.0,1,Madrid
    |4,Silla,Oficina,120.0,4,Barcelona
    |5,Mesa,Oficina,250.0,2,Sevilla
    |6,Laptop,Tecnología,1200.0,1,Madrid
    |7,Tablet,Tecnología,600.0,2,Bilbao
    |8,Ratón,Tecnología,25.0,10,Madrid
    |9,Impresora,Tecnología,200.0,1,Sevilla
    |10,Silla,Oficina,120.0,3,Madrid
    |11,Mesa,Oficina,250.0,1,Barcelona
    |12,Laptop,Tecnología,1200.0,1,Valencia
    |13,Monitor,Tecnología,350.0,2,Madrid
    |14,Tablet,Tecnología,600.0,1,Sevilla
    |15,Ratón,Tecnología,25.0,8,Bilbao""".stripMargin

val clientesCSV =
  """id,nombre,ciudad,segmento
    |,Ana García,Madrid,Premium
    |2,Pedro López,Valencia,Estándar
    |3,María Ruiz,Barcelona,Premium
    |4,,Sevilla,Estándar
    |5,Carlos Díaz,Bilbao,Premium
    |6,Laura Sanz,Madrid,Estándar
    |7,José Martín,Valencia,
    |8,Isabel Pérez,Madrid,Premium""".stripMargin

Files.write(Paths.get(s"$carpeta/ventas.csv"),   ventasCSV.getBytes(StandardCharsets.UTF_8))
Files.write(Paths.get(s"$carpeta/clientes.csv"), clientesCSV.getBytes(StandardCharsets.UTF_8))

println(s"✅ Ficheros creados en$carpeta")

✅ Ficheros creados enC:/Curso-Scala/datos/dia23


carpeta: String = "C:/Curso-Scala/datos/dia23"
res3_1: java.nio.file.Path = C:\Curso-Scala\datos\dia23
ventasCSV: String = """id,producto,categoria,precio,cantidad,ciudad
1,Laptop,Tecnología,1200.0,2,Madrid
2,Teclado,Tecnología,45.0,5,Valencia
3,Monitor,Tecnología,350.0,1,Madrid
4,Silla,Oficina,120.0,4,Barcelona
5,Mesa,Oficina,250.0,2,Sevilla
6,Laptop,Tecnología,1200.0,1,Madrid
7,Tablet,Tecnología,600.0,2,Bilbao
8,Ratón,Tecnología,25.0,10,Madrid
9,Impresora,Tecnología,200.0,1,Sevilla
10,Silla,Oficina,120.0,3,Madrid
11,Mesa,Oficina,250.0,1,Barcelona
12,Laptop,Tecnología,1200.0,1,Valencia
13,Monitor,Tecnología,350.0,2,Madrid
14,Tablet,Tecnología,600.0,1,Sevilla
15,Ratón,Tecnología,25.0,8,Bilbao"""
clientesCSV: String = """id,nombre,ciudad,segmento
,Ana García,Madrid,Premium
2,Pedro López,Valencia,Estándar
3,María Ruiz,Barcelona,Premium
4,,Sevilla,Estándar
5,Carlos Díaz,Bilbao,Premium
6,Laura Sanz,Madrid,Estándar
7,José Martín,Valencia,
8,Isabel Pérez,Madrid,Premium"""
res3_4: java.nio.fi

## P1 — Funciones de transformación reutilizables
Cada función hace UNA sola cosa y recibe/devuelve un DataFrame.

In [4]:
// P1 — Transformaciones como funciones DataFrame => DataFrame
val dfVentas = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("C:/Curso-Scala/datos/dia23/ventas.csv")

println("=== DataFrame original ===")
dfVentas.show(5)

def agregarImporte(df: DataFrame): DataFrame =
  df.withColumn("importe", col("precio") * col("cantidad"))

def soloTecnologia(df: DataFrame): DataFrame =
  df.filter(col("categoria") === "Tecnología")

def seleccionarColumnas(df: DataFrame): DataFrame =
  df.select("producto", "ciudad", "importe")

// Pipeline: encadenar las tres funciones
val dfResultado = seleccionarColumnas(soloTecnologia(agregarImporte(dfVentas)))

println("=== Resultado del pipeline ===")
dfResultado.show()

=== DataFrame original ===
+---+--------+----------+------+--------+---------+
| id|producto| categoria|precio|cantidad|   ciudad|
+---+--------+----------+------+--------+---------+
|  1|  Laptop|Tecnología|1200.0|       2|   Madrid|
|  2| Teclado|Tecnología|  45.0|       5| Valencia|
|  3| Monitor|Tecnología| 350.0|       1|   Madrid|
|  4|   Silla|   Oficina| 120.0|       4|Barcelona|
|  5|    Mesa|   Oficina| 250.0|       2|  Sevilla|
+---+--------+----------+------+--------+---------+
only showing top 5 rows
=== Resultado del pipeline ===
+---------+--------+-------+
| producto|  ciudad|importe|
+---------+--------+-------+
|   Laptop|  Madrid| 2400.0|
|  Teclado|Valencia|  225.0|
|  Monitor|  Madrid|  350.0|
|   Laptop|  Madrid| 1200.0|
|   Tablet|  Bilbao| 1200.0|
|    Ratón|  Madrid|  250.0|
|Impresora| Sevilla|  200.0|
|   Laptop|Valencia| 1200.0|
|  Monitor|  Madrid|  700.0|
|   Tablet| Sevilla|  600.0|
|    Ratón|  Bilbao|  200.0|
+---------+--------+-------+



dfVentas: DataFrame = [id: int, producto: string ... 4 more fields]
defined function agregarImporte
defined function soloTecnologia
defined function seleccionarColumnas
dfResultado: DataFrame = [producto: string, ciudad: string ... 1 more field]